# Training & Evaluation — Control (α = 0) vs Constrained (α = 1)

Trains both networks over several seeds, then evaluates them:

1. **Predictions + labels** — `predictions (N, T, 2)`, `labels (N,)`
2. **2D trajectories** of the predictions
3. **Hidden activations** — `(N, N_units, T)`
4. **Heatmaps** of the condition-averaged, peak-sorted activity
5. **RSM** matrices, control and constrained
6. **RSM** plots
7. **Boxplot — Epochs**
8. **Boxplot — Correlation**

Every function comes from `config.py`, `utils.py`, `data.py`, `model.py`,
`loading.py`, `analysis.py`, `plotting.py`. **No function is defined in this
notebook.**

## **Configuration**

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.stats import ttest_rel

from config import *        # constants + KWARGS_DATA / KWARGS_MODEL / KWARGS_LOAD_AND_PREDICT
from utils import *         # CustomReLU, CustomMSE, make_dataset
from data import *          # generate_diverging_trajectories
from model import *         # Model_1
from loading import *       # KWARGS_MODEL_PATH, model_path, load_and_predict
from analysis import *      # compute_rsm
from plotting import *      # plot_trajectories_2d, plot_sorted_averaged_activations,
                            # plot_rsm, boxplot_with_lines, plot_siginficance_symbol_paired

print_config(settings())
print("GPUs:", tf.config.list_physical_devices("GPU"))

### Run-level knobs

`QUICK = True` gives a short smoke run. Set it to `False` for the paper settings
(`NUM_RUNS = 20`, `MAX_TRAINING_TRIALS = 200_000`, `AUGMENTATION_FACTOR = 13`).

In [ ]:
QUICK = True

N_RUNS    = 3     if QUICK else NUM_RUNS
MAX_STEPS = 2_000 if QUICK else MAX_TRAINING_TRIALS
AUGMENT   = 11    if QUICK else AUGMENTATION_FACTOR
BATCH     = 1_000 if QUICK else BATCH_SIZE

##### alpha of the constrained net: alpha * K(K-1)/2, as in Training.txt,
##### because tf_corr_penalty returns the MEAN over the K(K-1)/2 pairs
FACTOR = NUM_TRAJECTORIES * (NUM_TRAJECTORIES - 1) / 2
ALPHAS_USED = (0, 1)

MODELS_DIR = "Models"
CKPT_DIR   = "ckpts"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"{N_RUNS} runs x 2 networks | <= {MAX_STEPS} steps | batch {BATCH} | "
      f"K = {NUM_TRAJECTORIES} | T = {NUM_STEPS} | alpha_constrained = 1 x {FACTOR:g}")

## **Data**

In [ ]:
KWARGS_DATA["plot"] = True
KWARGS_DATA["augmentation_factor"] = AUGMENT

INPUTS, OUTPUTS, angles = generate_diverging_trajectories(**KWARGS_DATA)

##### held-out evaluation batch, train on the rest  (as in Training.txt)
I_val, O_val = INPUTS[:BATCH], OUTPUTS[:BATCH]
I_train, O_train = INPUTS[BATCH:], OUTPUTS[BATCH:]

print("INPUTS :", INPUTS.shape, "| OUTPUTS :", OUTPUTS.shape)
print("train  :", I_train.shape, O_train.shape)
print("eval   :", I_val.shape, O_val.shape)
print("conditions :", np.unique(I_val))

## **Training**

Both networks of a run share the seed, so they share the connectivity mask and the
initial weights and differ only in the correlation penalty. Per run and per model we
keep **Epochs** (`len(lc[0])`, the number of optimisation steps, as in `Training.txt`)
and **Correlation** (`model.tf_corr_penalty` on the held-out batch).

In [ ]:
EPOCHS = {name: [] for name in MODEL_LABELS}
CORRS  = {name: [] for name in MODEL_LABELS}
PATHS  = {name: [] for name in MODEL_LABELS}
LOSSES = {name: [] for name in MODEL_LABELS}

for run in range(1, N_RUNS + 1):
    for name, alpha in zip(MODEL_LABELS, ALPHAS_USED):

        tf.keras.utils.set_random_seed(RANDOM_SEED + run)

        ##### build
        model = Model_1(**{**KWARGS_MODEL,
                           "alpha": alpha * FACTOR,
                           "num_steps": OUTPUTS.shape[1],
                           "seed": RANDOM_SEED + run})
        model.build(input_shape=(None, 1))

        print("-" * 62)
        print(f"  run {run}/{N_RUNS}  |  {name}  |  alpha = {float(model.alpha.numpy()):.3f}")
        print("-" * 62)

        ##### train
        lc, n_steps, losses_log = model.train(
            INPUTS=I_train, OUTPUTS=O_train,
            batch_size=BATCH,
            correlation_penalty=True,
            behavioral_loss_threshold=BEHAVIORAL_LOSS_THRESHOLD,
            MAX_TRAINING_TRIALS=MAX_STEPS,
            LOG_EVERY=LOG_EVERY,
            CKPT_EVERY=10 ** 9,
            ckpt_dir=f"{CKPT_DIR}/run_{run}_alpha_{alpha}",
            save_name=f"Penalized_run_{run}_alpha_{alpha}",
        )

        ##### save the weights under the project's naming convention
        MODEL_PATH = model_path(**{**KWARGS_MODEL_PATH,
                                   "K": NUM_TRAJECTORIES, "angle": ANGLE,
                                   "run": run, "alpha": alpha,
                                   "models_dir": MODELS_DIR})
        model.save_weights(MODEL_PATH)

        ##### metrics of this run
        I_t = tf.cast(I_val, model.data_dtype)
        preds, hidden_s = model(I_t)

        EPOCHS[name].append(len(lc[0]))
        CORRS[name].append(float(model.tf_corr_penalty(I_t, hidden_s).numpy()))
        LOSSES[name].append(np.asarray(lc[0], dtype=float))
        PATHS[name].append(MODEL_PATH)

        print(f"  epochs = {EPOCHS[name][-1]}  |  correlation = {CORRS[name][-1]:.4f}")
        print(f"  saved  : {MODEL_PATH}\n")

        del model, preds, hidden_s

print("Epochs      :", EPOCHS)
print("Correlation :", CORRS)

## **1. Predictions with their labels &nbsp;·&nbsp; 3. Hidden activations**

Both networks of run 1 are re-loaded from disk with `load_and_predict` and run on the
held-out batch.

* `PREDICTIONS[name]` → `(N, T, 2)`
* `LABELS` → `(N,)`
* `HIDDEN[name]` → `(N, N_units, T)`

In [ ]:
RUN_SHOWN = 1

PREDICTIONS = {}
HIDDEN      = {}
MODELS      = {}

LABELS = I_val.reshape(-1)                      # (N,)

for name, alpha in zip(MODEL_LABELS, ALPHAS_USED):

    preds, hidden_s, model = load_and_predict(**{**KWARGS_LOAD_AND_PREDICT,
                                                 "I": I_val,
                                                 "K": NUM_TRAJECTORIES,
                                                 "num_steps": OUTPUTS.shape[1],
                                                 "batch_size": BATCH,
                                                 "alpha": alpha * FACTOR,
                                                 "model_name": PATHS[name][RUN_SHOWN - 1]})

    PREDICTIONS[name] = preds.numpy()                                 # (N, T, 2)
    HIDDEN[name]      = np.transpose(hidden_s.numpy(), (0, 2, 1))     # (N, N_units, T)
    MODELS[name]      = model

    print(f"{name:12s} | predictions {PREDICTIONS[name].shape} | "
          f"hidden {HIDDEN[name].shape}")

print(f"{'labels':12s} | {LABELS.shape} | values {np.unique(LABELS)}")

## **2. 2D trajectories of the predictions**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.subplots_adjust(wspace=0.3)

plot_trajectories_2d(axes[0], O_val, I_val, title="Original Data",
                     legend_kwargs={"loc": "lower left", "bbox_to_anchor": (0.8, 0.05)})

plot_trajectories_2d(axes[1], PREDICTIONS[MODEL_LABELS[0]], I_val, title=MODEL_LABELS[0],
                     legend_kwargs={"loc": "lower left", "bbox_to_anchor": (0.8, 0.05)})

plot_trajectories_2d(axes[2], PREDICTIONS[MODEL_LABELS[1]], I_val, title=MODEL_LABELS[1],
                     legend_kwargs={"loc": "lower left", "bbox_to_anchor": (0.8, 0.05)})

plt.show()

## **4. Heatmaps**

`plot_sorted_averaged_activations` averages within condition, normalises each unit and
sorts the units once by their peak time in condition 1, then reuses that order for every
condition. It returns the sorted, condition-averaged activity `(K, N_units, T)`, which
the RSM below is computed from.

In [ ]:
ACT = {}
vmin, vmax = 0.6, 0.95

for name in MODEL_LABELS:
    fig, ax = plt.subplots(1, NUM_TRAJECTORIES, figsize=(15, 4))
    Labels, ACT[name] = plot_sorted_averaged_activations(ax, LABELS, HIDDEN[name],
                                                         vmin=vmin, vmax=vmax)
    fig.suptitle(name, y=1.03, fontweight="bold")
    plt.show()
    print(f"{name:12s} | conditions {Labels} | averaged activity {ACT[name].shape}")

## **5. RSM matrices &nbsp;·&nbsp; 6. RSM plots**

In [ ]:
RSM = {}

for name in MODEL_LABELS:
    RSM[name] = compute_rsm(ACT[name])
    print(f"{name:12s} | RSM {RSM[name].shape}")
    print(np.round(RSM[name], 3), "\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.subplots_adjust(wspace=0.4)

plot_rsm(RSM[MODEL_LABELS[0]], title=MODEL_LABELS[0], ax=axes[0], cbar_label="|r|")
plot_rsm(RSM[MODEL_LABELS[1]], title=MODEL_LABELS[1], ax=axes[1], cbar_label="|r|")

plt.show()

## **7. Boxplot — Epochs**

In [ ]:
data = np.array([EPOCHS[MODEL_LABELS[0]], EPOCHS[MODEL_LABELS[1]]], dtype=float)
padding = 0.2 * (data.max() - data.min() + 1e-12)

fig, ax = plt.subplots(figsize=(3, 5))
boxplot_with_lines(ax, data, xlabel=MODEL_LABELS, ylabel="Epochs", colours=PAIR,
                   ymin=data.min() - padding, ymax=data.max() + padding,
                   use_legend=False)

stat = {}
stat["t"], stat["p"] = ttest_rel(data[0], data[1])
plot_siginficance_symbol_paired(ax, data, stat, height=0.1)
plt.show()

print(f"Control     : {data[0].mean():.1f} +/- {data[0].std(ddof=1):.1f}")
print(f"Constrained : {data[1].mean():.1f} +/- {data[1].std(ddof=1):.1f}")
print(f"t = {stat['t']}, p = {stat['p']}, n = {data.shape[1]}")

## **8. Boxplot — Correlation**

In [ ]:
data = np.array([CORRS[MODEL_LABELS[0]], CORRS[MODEL_LABELS[1]]], dtype=float)
padding = 0.2 * (data.max() - data.min() + 1e-12)

fig, ax = plt.subplots(figsize=(3, 5))
boxplot_with_lines(ax, data, xlabel=MODEL_LABELS, ylabel="Cross-condition |r|",
                   colours=PAIR,
                   ymin=data.min() - padding, ymax=data.max() + padding,
                   use_legend=False)

stat = {}
stat["t"], stat["p"] = ttest_rel(data[0], data[1])
plot_siginficance_symbol_paired(ax, data, stat, height=0.1)
plt.show()

print(f"Control     : {data[0].mean():.4f} +/- {data[0].std(ddof=1):.4f}")
print(f"Constrained : {data[1].mean():.4f} +/- {data[1].std(ddof=1):.4f}")
print(f"t = {stat['t']}, p = {stat['p']}, n = {data.shape[1]}")